# NYC Subway Ridership Forecasting

This notebook analyzes historical MTA subway ridership data (2020-2026) and builds forecasting models
to predict near-term ridership trends. The data is sourced from the MTA's published ridership statistics,
stored as Parquet files in S3.

**Goals:**
- Explore ridership trends through the COVID-19 pandemic and recovery period
- Identify day-of-week, seasonal, and holiday patterns
- Build and compare two forecasting models: Facebook Prophet and XGBoost
- Generate a 30-day forward forecast with uncertainty intervals
- Evaluate model accuracy and identify the most predictive features

In [ ]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
from prophet import Prophet
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 120

%matplotlib inline

In [ ]:
# ---- Configuration ----
# Replace YOUR_ACCOUNT_ID with your actual AWS account ID
ANALYTICS_BUCKET = "railtime-analytics-YOUR_ACCOUNT_ID"
ML_BUCKET = "railtime-ml-YOUR_ACCOUNT_ID"

RIDERSHIP_PATH = f"s3://{ANALYTICS_BUCKET}/historical/daily_ridership/"
MODEL_OUTPUT_PATH = f"s3://{ML_BUCKET}/models/ridership/"

# COVID-19 key dates for annotations
COVID_LOCKDOWN_DATE = pd.Timestamp("2020-03-22")
PHASE_1_REOPEN = pd.Timestamp("2020-06-08")
PHASE_4_REOPEN = pd.Timestamp("2020-07-20")
VACCINE_AVAILABLE = pd.Timestamp("2021-04-06")
MASK_MANDATE_LIFTED = pd.Timestamp("2022-09-07")

# Federal holidays (US) for feature engineering
FEDERAL_HOLIDAYS = [
    "2020-01-01", "2020-01-20", "2020-02-17", "2020-05-25", "2020-07-03",
    "2020-09-07", "2020-10-12", "2020-11-11", "2020-11-26", "2020-12-25",
    "2021-01-01", "2021-01-18", "2021-02-15", "2021-05-31", "2021-07-05",
    "2021-09-06", "2021-10-11", "2021-11-11", "2021-11-25", "2021-12-24",
    "2022-01-01", "2022-01-17", "2022-02-21", "2022-05-30", "2022-07-04",
    "2022-09-05", "2022-10-10", "2022-11-11", "2022-11-24", "2022-12-26",
    "2023-01-02", "2023-01-16", "2023-02-20", "2023-05-29", "2023-07-04",
    "2023-09-04", "2023-10-09", "2023-11-10", "2023-11-23", "2023-12-25",
    "2024-01-01", "2024-01-15", "2024-02-19", "2024-05-27", "2024-07-04",
    "2024-09-02", "2024-10-14", "2024-11-11", "2024-11-28", "2024-12-25",
    "2025-01-01", "2025-01-20", "2025-02-17", "2025-05-26", "2025-07-04",
    "2025-09-01", "2025-10-13", "2025-11-11", "2025-11-27", "2025-12-25",
    "2026-01-01", "2026-01-19", "2026-02-16"
]
FEDERAL_HOLIDAYS = pd.to_datetime(FEDERAL_HOLIDAYS)

In [ ]:
# ---- Load Data ----
df = wr.s3.read_parquet(RIDERSHIP_PATH)

print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"\nNull counts:\n{df.isnull().sum()}")
df.head(10)

## Exploratory Data Analysis

We examine the overall ridership trajectory from 2020 through the present, identify day-of-week
cycling, and decompose the time series into trend, seasonal, and residual components.

In [ ]:
# ---- EDA: Daily Subway Ridership Trend (2020-2026) ----
ts = df.set_index("date").sort_index()

# Use 'subway_ridership' column -- adjust name to match your dataset
ridership_col = [c for c in ts.columns if "subway" in c.lower() and "ridership" in c.lower()]
if ridership_col:
    COL = ridership_col[0]
else:
    # Fallback: use first numeric column
    COL = ts.select_dtypes(include=[np.number]).columns[0]
    print(f"Warning: 'subway_ridership' column not found. Using '{COL}' instead.")

fig, ax = plt.subplots(figsize=(16, 7))

# 7-day rolling average for cleaner trend
ts["rolling_7d"] = ts[COL].rolling(7, min_periods=1).mean()

ax.plot(ts.index, ts[COL], alpha=0.25, linewidth=0.5, color="steelblue", label="Daily")
ax.plot(ts.index, ts["rolling_7d"], linewidth=1.5, color="navy", label="7-day Avg")

# Annotate key events
annotations = [
    (COVID_LOCKDOWN_DATE, "COVID Lockdown\n(Mar 22, 2020)", -60, 40),
    (PHASE_1_REOPEN, "Phase 1 Reopen\n(Jun 8, 2020)", 20, 50),
    (VACCINE_AVAILABLE, "Vaccine Widely\nAvailable (Apr 2021)", -80, 40),
    (MASK_MANDATE_LIFTED, "Mask Mandate\nLifted (Sep 2022)", 20, 40),
]

for date, label, xoff, yoff in annotations:
    if date >= ts.index.min() and date <= ts.index.max():
        y_val = ts.loc[ts.index >= date, "rolling_7d"].iloc[0] if date in ts.index or True else 0
        nearest_idx = ts.index[ts.index.searchsorted(date)]
        y_val = ts.loc[nearest_idx, "rolling_7d"]
        ax.annotate(
            label, xy=(date, y_val),
            xytext=(xoff, yoff), textcoords="offset points",
            arrowprops=dict(arrowstyle="->", color="red", lw=1.2),
            fontsize=8, ha="center", color="red", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="red", alpha=0.8)
        )

ax.set_title("NYC Subway Daily Ridership (2020-2026)", fontsize=15, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Estimated Ridership")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- EDA: Day-of-Week Patterns ----
dow_df = df.copy()
dow_df["day_of_week"] = pd.to_datetime(dow_df["date"]).dt.day_name()
dow_df["is_weekend"] = pd.to_datetime(dow_df["date"]).dt.dayofweek >= 5

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot by day of week
sns.boxplot(
    data=dow_df, x="day_of_week", y=COL, order=day_order,
    palette="coolwarm", ax=axes[0], fliersize=1
)
axes[0].set_title("Ridership Distribution by Day of Week", fontweight="bold")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=45)

# Weekend vs Weekday comparison
sns.boxplot(
    data=dow_df, x="is_weekend", y=COL,
    palette=["steelblue", "coral"], ax=axes[1], fliersize=1
)
axes[1].set_xticklabels(["Weekday", "Weekend"])
axes[1].set_title("Weekday vs Weekend Ridership", fontweight="bold")
axes[1].set_xlabel("")

# Add statistical summary
weekday_mean = dow_df.loc[~dow_df["is_weekend"], COL].mean()
weekend_mean = dow_df.loc[dow_df["is_weekend"], COL].mean()
axes[1].axhline(weekday_mean, color="steelblue", linestyle="--", alpha=0.7, label=f"Weekday mean: {weekday_mean:,.0f}")
axes[1].axhline(weekend_mean, color="coral", linestyle="--", alpha=0.7, label=f"Weekend mean: {weekend_mean:,.0f}")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Weekend ridership is {weekend_mean / weekday_mean * 100:.1f}% of weekday ridership.")

In [ ]:
# ---- EDA: Seasonal Decomposition ----
# Resample to daily frequency, forward-fill any gaps
daily = ts[[COL]].resample("D").mean().ffill()

decomposition = seasonal_decompose(daily[COL], model="additive", period=365)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

components = [
    (decomposition.observed, "Observed", "steelblue"),
    (decomposition.trend, "Trend", "navy"),
    (decomposition.seasonal, "Seasonal", "green"),
    (decomposition.resid, "Residual", "red"),
]

for ax, (data, title, color) in zip(axes, components):
    ax.plot(data, color=color, linewidth=0.8)
    ax.set_ylabel(title, fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.3)

axes[0].set_title("Seasonal Decomposition of Subway Ridership (period=365 days)", fontsize=14, fontweight="bold")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

## Feature Engineering

We create temporal, rolling, and COVID-recovery features to power both forecasting models.

In [ ]:
# ---- Feature Engineering ----
feat = df.copy()
feat["date"] = pd.to_datetime(feat["date"])
feat = feat.sort_values("date").reset_index(drop=True)

# Temporal features
feat["day_of_week"] = feat["date"].dt.dayofweek  # 0=Monday, 6=Sunday
feat["month"] = feat["date"].dt.month
feat["day_of_year"] = feat["date"].dt.dayofyear
feat["week_of_year"] = feat["date"].dt.isocalendar().week.astype(int)
feat["is_weekend"] = (feat["day_of_week"] >= 5).astype(int)
feat["is_holiday"] = feat["date"].isin(FEDERAL_HOLIDAYS).astype(int)

# Rolling features
feat["rolling_7d"] = feat[COL].rolling(7, min_periods=1).mean()
feat["rolling_30d"] = feat[COL].rolling(30, min_periods=1).mean()
feat["rolling_7d_std"] = feat[COL].rolling(7, min_periods=1).std()

# Year-over-year change
feat["yoy_change"] = feat[COL].pct_change(periods=365)

# COVID recovery percentage (vs Feb 2020 baseline)
feb_2020_mask = (feat["date"].dt.year == 2020) & (feat["date"].dt.month == 2)
feb_2020_baseline = feat.loc[feb_2020_mask, COL].mean()
feat["covid_recovery_pct"] = feat[COL] / feb_2020_baseline * 100

# Lag features
for lag in [1, 7, 14, 28]:
    feat[f"lag_{lag}d"] = feat[COL].shift(lag)

# Cyclical encoding for day_of_week and month
feat["dow_sin"] = np.sin(2 * np.pi * feat["day_of_week"] / 7)
feat["dow_cos"] = np.cos(2 * np.pi * feat["day_of_week"] / 7)
feat["month_sin"] = np.sin(2 * np.pi * feat["month"] / 12)
feat["month_cos"] = np.cos(2 * np.pi * feat["month"] / 12)

# Drop rows with NaNs from rolling/lag
feat_clean = feat.dropna().reset_index(drop=True)

print(f"Feature matrix shape: {feat_clean.shape}")
print(f"\nFeatures created:")
print([c for c in feat_clean.columns if c not in df.columns])
print(f"\nFeb 2020 baseline ridership: {feb_2020_baseline:,.0f}")
print(f"Latest recovery %: {feat_clean['covid_recovery_pct'].iloc[-1]:.1f}%")
feat_clean.tail(5)

## Model 1: Facebook Prophet

Prophet handles trend changepoints automatically and allows us to explicitly mark COVID lockdown
and recovery as structural breaks in the time series.

In [ ]:
# ---- Prophet Model ----
prophet_df = feat_clean[["date", COL]].rename(columns={"date": "ds", COL: "y"})

# Define changepoints at major COVID milestones
changepoints = [
    "2020-03-22",  # NYC lockdown
    "2020-06-08",  # Phase 1 reopen
    "2020-09-01",  # Post-summer return
    "2021-04-06",  # Vaccine widely available
    "2021-09-13",  # Return to office push
    "2022-01-15",  # Omicron wave
    "2022-09-07",  # Mask mandate lifted
    "2023-09-01",  # Post-pandemic normal
]

model_prophet = Prophet(
    changepoints=changepoints,
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.15,
    seasonality_prior_scale=10.0,
)

# Add US holidays
model_prophet.add_country_holidays(country_name="US")

model_prophet.fit(prophet_df)

# Forecast 30 days into the future
future = model_prophet.make_future_dataframe(periods=30)
forecast = model_prophet.predict(future)

# Plot
fig, ax = plt.subplots(figsize=(16, 7))

# Historical
ax.plot(prophet_df["ds"], prophet_df["y"], ".", markersize=1, alpha=0.3, color="steelblue", label="Actual")

# Forecast
forecast_future = forecast[forecast["ds"] > prophet_df["ds"].max()]
forecast_hist = forecast[forecast["ds"] <= prophet_df["ds"].max()]

ax.plot(forecast_hist["ds"], forecast_hist["yhat"], linewidth=1, color="navy", alpha=0.6, label="Prophet Fit")
ax.plot(forecast_future["ds"], forecast_future["yhat"], linewidth=2, color="red", label="30-Day Forecast")
ax.fill_between(
    forecast_future["ds"],
    forecast_future["yhat_lower"],
    forecast_future["yhat_upper"],
    alpha=0.2, color="red", label="Uncertainty Interval"
)

ax.set_title("Prophet: 30-Day Ridership Forecast", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Estimated Ridership")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)

# Zoom to last 6 months + forecast
zoom_start = prophet_df["ds"].max() - pd.Timedelta(days=180)
ax.set_xlim(zoom_start, forecast_future["ds"].max() + pd.Timedelta(days=5))

plt.tight_layout()
plt.show()

# Prophet component plots
fig2 = model_prophet.plot_components(forecast)
plt.tight_layout()
plt.show()

## Model 2: XGBoost Regression

A gradient-boosted tree model using our engineered features. We split temporally, holding out the
last 60 days as the test set to simulate real forecasting conditions.

In [ ]:
# ---- XGBoost Model ----
FEATURE_COLS = [
    "day_of_week", "month", "day_of_year", "week_of_year",
    "is_weekend", "is_holiday",
    "rolling_7d", "rolling_30d", "rolling_7d_std",
    "lag_1d", "lag_7d", "lag_14d", "lag_28d",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "covid_recovery_pct",
]

X = feat_clean[FEATURE_COLS]
y = feat_clean[COL]

# Temporal split: last 60 days = test
split_idx = len(feat_clean) - 60
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = feat_clean["date"].iloc[split_idx:]

print(f"Train: {X_train.shape[0]} samples ({feat_clean['date'].iloc[0].date()} to {feat_clean['date'].iloc[split_idx-1].date()})")
print(f"Test:  {X_test.shape[0]} samples ({feat_clean['date'].iloc[split_idx].date()} to {feat_clean['date'].iloc[-1].date()})")

model_xgb = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    early_stopping_rounds=30,
)

model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

y_pred = model_xgb.predict(X_test)

# Plot actual vs predicted
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(dates_test.values, y_test.values, linewidth=1.5, color="steelblue", label="Actual")
ax.plot(dates_test.values, y_pred, linewidth=1.5, color="red", linestyle="--", label="XGBoost Predicted")
ax.fill_between(dates_test.values, y_test.values, y_pred, alpha=0.15, color="red")

ax.set_title("XGBoost: Actual vs Predicted (Last 60 Days)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Estimated Ridership")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Model Comparison: MAPE and RMSE ----

# Prophet metrics on the same test period
prophet_test = forecast.merge(
    prophet_df, on="ds", how="inner"
).tail(60)

prophet_mape = mean_absolute_percentage_error(prophet_test["y"], prophet_test["yhat"])
prophet_rmse = np.sqrt(mean_squared_error(prophet_test["y"], prophet_test["yhat"]))

# XGBoost metrics
xgb_mape = mean_absolute_percentage_error(y_test, y_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

comparison = pd.DataFrame({
    "Model": ["Prophet", "XGBoost"],
    "MAPE (%)": [prophet_mape * 100, xgb_mape * 100],
    "RMSE": [prophet_rmse, xgb_rmse],
})

print("\n" + "=" * 50)
print("MODEL COMPARISON (Test Period: Last 60 Days)")
print("=" * 50)
print(comparison.to_string(index=False, float_format="{:.2f}".format))
print("=" * 50)

winner = "Prophet" if prophet_mape < xgb_mape else "XGBoost"
print(f"\nBest model by MAPE: {winner}")

comparison

In [ ]:
# ---- XGBoost Feature Importance ----
importance = pd.Series(
    model_xgb.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importance.plot(kind="barh", ax=ax, color="steelblue", edgecolor="navy")
ax.set_title("XGBoost Feature Importance (Gain)", fontsize=14, fontweight="bold")
ax.set_xlabel("Feature Importance")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
for feat_name, score in importance.sort_values(ascending=False).head(5).items():
    print(f"  {feat_name}: {score:.4f}")

In [ ]:
# ---- Save Artifacts to S3 ----
import json
from datetime import datetime

run_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

# 1. Save 30-day forecast CSV
forecast_output = forecast_future[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
forecast_output.columns = ["date", "forecast", "lower_bound", "upper_bound"]
forecast_output["model"] = "prophet"
forecast_output["generated_at"] = run_timestamp

forecast_s3_path = f"s3://{ML_BUCKET}/models/ridership/forecasts/forecast_{run_timestamp}.csv"
wr.s3.to_csv(forecast_output, forecast_s3_path, index=False)
print(f"Forecast saved to: {forecast_s3_path}")

# 2. Save model evaluation metrics
metrics = {
    "run_timestamp": run_timestamp,
    "test_period_days": 60,
    "prophet": {"mape_pct": round(prophet_mape * 100, 2), "rmse": round(prophet_rmse, 2)},
    "xgboost": {"mape_pct": round(xgb_mape * 100, 2), "rmse": round(xgb_rmse, 2)},
    "best_model": winner,
    "feb_2020_baseline": round(feb_2020_baseline, 0),
    "latest_recovery_pct": round(feat_clean["covid_recovery_pct"].iloc[-1], 1),
}

metrics_path = f"s3://{ML_BUCKET}/models/ridership/metrics/metrics_{run_timestamp}.json"
wr.s3.to_json(pd.DataFrame([metrics]), metrics_path, orient="records")
print(f"Metrics saved to: {metrics_path}")

# 3. Save XGBoost model
import tempfile
import boto3

with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as tmp:
    model_xgb.save_model(tmp.name)
    s3_client = boto3.client("s3")
    model_key = f"models/ridership/artifacts/xgboost_{run_timestamp}.json"
    s3_client.upload_file(tmp.name, ML_BUCKET, model_key)
    print(f"XGBoost model saved to: s3://{ML_BUCKET}/{model_key}")

# 4. Save Prophet model
with tempfile.NamedTemporaryFile(suffix=".json", delete=False, mode="w") as tmp:
    from prophet.serialize import model_to_json
    tmp.write(model_to_json(model_prophet))
    tmp.flush()
    model_key = f"models/ridership/artifacts/prophet_{run_timestamp}.json"
    s3_client.upload_file(tmp.name, ML_BUCKET, model_key)
    print(f"Prophet model saved to: s3://{ML_BUCKET}/{model_key}")

print(f"\nAll artifacts saved successfully for run: {run_timestamp}")